# 01 · Raw EDA

Goal: understand the raw files before building the pipeline.

## 1. Files

Which files are in the raw folder?

In [1]:
from pathlib import Path

import pandas as pd

RAW = Path("../data/raw/elliptic/elliptic_bitcoin_dataset")
print([p.name for p in RAW.iterdir()])

['elliptic_txs_edgelist.csv', 'elliptic_txs_features.csv', 'elliptic_txs_classes.csv']


Three files:

- `elliptic_txs_classes.csv`
- `elliptic_txs_edgelist.csv`
- `elliptic_txs_features.csv`

## 2. Classes

What does the classes file look like?

In [2]:
classes = pd.read_csv(RAW / "elliptic_txs_classes.csv")
print(classes.shape)
print(classes.dtypes)
print(classes.head())

(203769, 2)
txId     int64
class      str
dtype: object


203,769 rows and 2 columns:

- `txId`: integer.
- `class`: text.

The first rows show `class` with values like `unknown` and `2`.

### 2.1 Values of `class`

Which values does `class` have, how many of each, and what share of the rows? Empty values are counted too, and the counts are checked against the number of rows.

In [3]:
counts = classes["class"].value_counts(dropna=False)
percent = (counts / len(classes) * 100).round(2)
print(pd.DataFrame({"count": counts, "percent": percent}))
print()
print("sum of counts:", counts.sum())
print("rows:", len(classes))

          count  percent
class                   
unknown  157205    77.15
2         42019    20.62
1          4545     2.23

sum of counts: 203769
rows: 203769


`class` has three values:

- `unknown`: 157,205 (77.15%)
- `2`: 42,019 (20.62%)
- `1`: 4,545 (2.23%)

No empty values. The counts add up to 203,769, the number of rows.

### 2.2 Duplicate ids

Does any `txId` appear more than once?

In [4]:
print(classes["txId"].duplicated().sum())

0


No repeated `txId`. Each row is a different transaction.

## 3. Edges

What does the edges file look like?

In [5]:
edges = pd.read_csv(RAW / "elliptic_txs_edgelist.csv")
print(edges.shape)
print(edges.dtypes)
print(edges.head())

(234355, 2)
txId1    int64
txId2    int64
dtype: object


234,355 rows and 2 columns:

- `txId1`: integer.
- `txId2`: integer.

Both columns look like the `txId` values from `classes`.

### 3.1 Values of `txId1` and `txId2`

Are there empty values, and how many different ids does each column have?

In [6]:
print(edges.isna().sum())
print()
print(edges.nunique())

txId1    0
txId2    0
dtype: int64

txId1    166345
txId2    148447
dtype: int64


No empty values in `txId1` or `txId2`.

Different ids per column:

- `txId1`: 166,345
- `txId2`: 148,447

### 3.2 Duplicate edges

Does any pair (`txId1`, `txId2`) appear more than once?

In [7]:
print(edges.duplicated().sum())

0


No repeated pair. Each row is a different edge.

### 3.3 Self-loops

Does any edge have the same id in `txId1` and `txId2`?

In [8]:
print((edges["txId1"] == edges["txId2"]).sum())

0


No self-loops. Every edge connects two different transactions.

## 4. Features

What does the features file look like?

In [9]:
features = pd.read_csv(RAW / "elliptic_txs_features.csv", header=None)
print(features.shape)
print(features.iloc[:5, :6])

(203769, 167)


203,769 rows and 167 columns. There is no header, so the columns are numbered 0 to 166.

- Column 0: looks like `txId`. The first 5 values are the same as in `classes`.
- Column 1: integer, `1` in these first rows.
- Columns 2 to 166: decimal numbers.

### 4.1 Column types

How many columns of each type are there?

In [10]:
print(features.dtypes.value_counts())

float64    165
int64        2
Name: count, dtype: int64


- 2 columns are integers (`int64`).
- 165 columns are decimals (`float64`).

### 4.2 Which columns are integers

Which column numbers have type `int64`?

In [11]:
print(features.columns[features.dtypes == "int64"].tolist())

[0, 1]


The integer columns are 0 and 1. All the other columns (2 to 166) are decimals.

### 4.3 Values of column 1

Which values does column 1 have, and how many rows of each?

In [12]:
counts = features[1].value_counts(dropna=False).sort_index()
print(counts)
print()
print("sum of counts:", counts.sum())
print("rows:", len(features))

1
1     7880
2     4544
3     6621
4     5693
5     6803
6     4328
7     6048
8     4457
9     4996
10    6727
11    4296
12    2047
13    4528
14    2022
15    3639
16    2975
17    3385
18    1976
19    3506
20    4291
21    3537
22    5894
23    4165
24    4592
25    2314
26    2523
27    1089
28    1653
29    4275
30    2483
31    2816
32    4525
33    3151
34    2486
35    5507
36    6393
37    3306
38    2891
39    2760
40    4481
41    5342
42    7140
43    5063
44    4975
45    5598
46    3519
47    5121
48    2954
49    2454
Name: count, dtype: int64

sum of counts: 203769
rows: 203769


Column 1 has 49 values, from 1 to 49, with no empty values. Rows per value range from 1,089 (value 27) to 7,880 (value 1).

The counts add up to 203,769, the number of rows.

### 4.4 Column 0: ids

Does any value in column 0 appear more than once, and are there empty values?

In [13]:
print("empty:", features[0].isna().sum())
print("repeated:", features[0].duplicated().sum())

empty: 0
repeated: 0


Column 0 has no empty values and no repeated values. Each row is a different transaction.

### 4.5 Empty values in the decimal columns

Are there empty values in columns 2 to 166?

In [14]:
print(features.isna().sum().sum())

0


No empty values in the whole file, in any of the 167 columns.

## 5. Across files

Do the three files describe the same transactions?

### 5.1 `features` and `classes`

Are the ids in column 0 of `features` the same as the `txId` in `classes`?

In [15]:
print(set(features[0]) == set(classes["txId"]))

True


The ids in `features` (column 0) and the `txId` in `classes` are exactly the same set: 203,769 transactions.

### 5.2 `edges` and `classes`

Do all ids in `edges` exist in `classes`?

In [16]:
ids = set(classes["txId"])
print("txId1 not in classes:", (~edges["txId1"].isin(ids)).sum())
print("txId2 not in classes:", (~edges["txId2"].isin(ids)).sum())

txId1 not in classes: 0
txId2 not in classes: 0


All ids in `txId1` and `txId2` exist in `classes`. No edge points to an unknown transaction.

### 5.3 `edges` and `features`: time steps

Do both ends of an edge belong to the same time step?

In [17]:
time_step = features.set_index(0)[1]
same_step = edges["txId1"].map(time_step) == edges["txId2"].map(time_step)
print(same_step.value_counts())

True    234355
Name: count, dtype: int64


All 234,355 edges connect two transactions in the same time step. Each time step is a separate graph.

### 5.4 `edges` and `classes`: isolated transactions

Does every transaction appear in at least one edge?

In [18]:
n = len(classes)
print(
    "no edge at all:",
    len(set(classes["txId"]) - set(edges["txId1"]) - set(edges["txId2"])),
)
print("no outgoing edge:", n - edges["txId1"].nunique())
print("no incoming edge:", n - edges["txId2"].nunique())

no edge at all: 0
no outgoing edge: 37424
no incoming edge: 55322


Every transaction appears in at least one edge, so none is isolated.

- 37,424 transactions have no outgoing edge.
- 55,322 transactions have no incoming edge.

## 6. Summary

- `classes`: 203,769 rows. `class` is `unknown` (77.15%), `2` (20.62%) or `1` (2.23%). `txId` is unique.
- `edges`: 234,355 rows. No empty values, repeated pairs or self-loops.
- `features`: 203,769 rows, 167 columns, no header. Column 0 is the id, column 1 is a number from 1 to 49, columns 2 to 166 are decimals. No empty values.
- Across files: `features` and `classes` have the same ids. Every id in `edges` exists in `classes`. Every edge connects two transactions in the same time step. Every transaction appears in at least one edge (37,424 have no outgoing edge and 55,322 have no incoming edge).